Run this first — it installs everything the lab needs.

In [ ]:
%pip install -q qiskit qiskit-aer qiskit-addon-cutting pylatexenc matplotlib

# 19. Quantum Error Mitigation II

## Objectives
* Implement four more mitigation tools by hand — **circuit optimization**, **Pauli twirling**, **readout-error correction**, and **zero-noise extrapolation** — and meet **circuit knitting**.
* See that *every* mitigation buys accuracy with a **resource**: extra shots, runtime, money, or "wait for better hardware."
* Use a resource model to find the *cheapest*, *fastest*, or *soonest* way to get a trustworthy answer out of a noisy machine.

In **Lab 18** you turned coherent error into well-behaved stochastic error with Pauli
twirling. Back in **Lab 9** you saw a different kind of trade-off: a gate's *synthesis*
error (how well a finite circuit approximates it) fights its *hardware* error (every gate
you run is noisy), and the best circuit isn't always the most accurate one.

This lab puts those two ideas together and adds a price tag. You'll be handed one noisy
computation and a pile of mitigation techniques, and your job is to get the right answer
**as cheaply, quickly, or soon as possible** — because on real hardware, "just make it
perfect" is never one of the options.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import (Pauli, Operator, Statevector, DensityMatrix,
                                 state_fidelity, Clifford)
from qiskit.circuit.library import CXGate
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (NoiseModel, depolarizing_error,
                              thermal_relaxation_error)

np.set_printoptions(suppress=True, precision=4)

N = 8                         # qubits
OBS = Pauli("X" * N)          # the observable we care about: <X_0 X_1 ... X_7>
TG1, TG2 = 50e-9, 300e-9      # 1- and 2-qubit gate times (seconds), for T1/T2 errors

# The single hardware knob. s = 1.0 is "today"; smaller s = cleaner (future) hardware.
# Everything about the noise is derived from this one number.
def p2_of(s):
    # physical two-qubit gate error rate as a function of s (sub-linear: better
    # hardware costs a lot of s). Today (s=1) -> 1% per CNOT.
    return 0.01 * s**0.5

## 1. The patient

Here is the computation you've been handed: an **8-qubit, 40-layer** circuit. Its job is
to prepare a *GHZ state* $\tfrac{1}{\sqrt2}(|00000000\rangle + |11111111\rangle)$ — the
maximally entangled "all agree" state — and the number we want out of it is the
expectation value of the global parity in the $X$ basis,

$$\langle O\rangle = \langle X_0 X_1 \cdots X_7\rangle.$$

For a perfect GHZ state this is exactly $\mathbf{+1}$. It's also a genuine quantum
quantity: the state is a superposition, so a single shot tells you almost nothing — you
have to average over **many** shots to estimate $\langle O\rangle$. And because $\langle O\rangle$
is so sensitive to *everything* going wrong (a single bit-flip or lost coherence drags it
toward 0), it's a brutal, honest scorecard for how much noise ate your circuit.

Run the cell to meet the circuit.

In [ ]:
def patient_circuit():
    qc = QuantumCircuit(N)
    qc.h(0)
    for q in range(N - 1):
        qc.cx(q, q + 1)
        qc.h(q); qc.h(q)            # <- redundant pair
        qc.x(q + 1); qc.x(q + 1)    # <- redundant pair
    for _ in range(3):
        for q in range(0, N - 1, 2):
            qc.cx(q, q + 1); qc.cx(q, q + 1)        # <- redundant pair
        for q in range(N):
            qc.rz(0.37, q); qc.rz(-0.37, q)         # <- cancel to a phase
        for q in range(1, N - 1, 2):
            qc.h(q); qc.cx(q, q + 1); qc.h(q)       # <- two of these in a row...
            qc.h(q); qc.cx(q, q + 1); qc.h(q)       #     ...hide an identity
    return qc

RAW = patient_circuit()
print("depth:", RAW.depth(), " | CNOTs:", RAW.count_ops().get("cx", 0))

Before running anything, take a good look at the circuit you've been handed. It's
deliberately **messy** — keep an eye out for gates that obviously undo each other (an `H`
right after an `H`, an `X` after an `X`, and so on). You'll be the one cleaning them up in
section 2, so it pays to see the junk now.

In [ ]:
# A visual tour of the patient. Scan for gates that cancel — that's free fidelity later.
RAW.draw("mpl", fold=28)

First, the target we're shooting for. On a *perfect* machine the answer is fixed by the
math — no noise, no shots. Compute it directly from the statevector so we know what
"right" looks like.

<b>Compute the ideal $\langle O\rangle$.</b> `Statevector(circuit)` gives the exact output
state; `.expectation_value(OBS)` returns $\langle O\rangle$. (It will be real; wrap in
`np.real`.)

In [ ]:
# your code here
???

Now the hardware. A real device has several error channels at once, and we'll bottle them
all into one `NoiseModel` controlled by the single knob `s`:

* **Depolarizing error** on every gate — a chance the gate just scrambles the qubit(s).
  Two-qubit gates are ~10× worse than one-qubit gates (`p2_of(s)` vs `p2_of(s)/10`).
* **T1/T2 relaxation** — qubits decay (`T1`) and dephase (`T2`) while gates take time.
  Cleaner hardware has *longer* coherence, so `T1, T2` scale up as `s` shrinks.

The thermal-relaxation pieces are built for you. <b>Add the depolarizing errors and
register everything in the model.</b> Build `e1` = 1-qubit depolarizing composed with
`relax1`, `e2` = 2-qubit depolarizing composed with `relax2`, then attach `e1` to the
1-qubit gates and `e2` to `"cx"`.

In [ ]:
def make_noise_model(s):
    q = p2_of(s)                          # 2-qubit depolarizing rate
    t1, t2 = 2.5e-6 / q, 1.5e-6 / q       # coherence times (longer = cleaner hardware)
    relax1 = thermal_relaxation_error(t1, t2, TG1)
    relax2 = thermal_relaxation_error(t1, t2, TG2).tensor(
             thermal_relaxation_error(t1, t2, TG2))
    nm = NoiseModel()
    # your code here
    ???
    return nm

# Readout (measurement) error is kept separate — you'll model and undo it in section 4.
def readout_rate(s):
    return p2_of(s) / 2                   # probability a measured bit is flipped

In [ ]:
# Provided tools — you do not need to change these.
_dm = AerSimulator(method="density_matrix")

def expval(circuit, s):
    # Exact <O> after running `circuit` at hardware level s (the many-shot limit,
    # gate noise only). This is what an infinitely patient experimenter would measure
    # before readout error.
    qc = circuit.copy(); qc.save_density_matrix()
    t = transpile(qc, _dm, optimization_level=0,
                  basis_gates=["u", "cx", "save_density_matrix"])
    rho = _dm.run(t, noise_model=make_noise_model(s)).result().data()["density_matrix"]
    return float(np.real(DensityMatrix(rho).expectation_value(OBS)))

_shot = AerSimulator()

def sample_O(circuit, s, shots):
    # Estimate <O> from a finite number of shots (includes readout error), so you can
    # see why many shots are needed.
    qc = circuit.copy()
    qc.h(range(N))            # rotate X-basis onto the computational basis
    qc.measure_all()
    t = transpile(qc, _shot, optimization_level=0)
    counts = _shot.run(t, noise_model=make_noise_model(s), shots=shots,
                       seed_simulator=7).result().get_counts()
    total = even = 0
    for bits, c in counts.items():
        parity = bits.replace(" ", "").count("1") % 2     # X-basis parity of this shot
        total += c
        if parity == 0: even += c
    return (even - (total - even)) / total                # <O> = P(even) - P(odd)

Time to see the damage. We start at **today's** hardware, `s = 1.0`.

<b>Estimate $\langle O\rangle$ on the raw circuit at `s = 1.0`.</b> First try a stingy
`shots = 20` and then a generous `shots = 20000`, using `sample_O(RAW, 1.0, shots)`. Notice
two things: with few shots the estimate is pure noise (the state is entangled — one shot
is almost useless), and even with many shots the answer is nowhere near the ideal $+1$.

In [ ]:
# your code here
???

## 2. The cheapest mitigation is not making the error

Before any fancy technique, remember **Lab 9**: every CNOT you run costs fidelity, so the
single most powerful thing you can do is *run fewer gates*. Look back at the circuit you
just drew — it is stuffed with junk:

* `H` then `H` (cancels), `X` then `X` (cancels), `CX` then `CX` (cancels),
* `Rz(0.37)` then `Rz(-0.37)` (cancels), and
* an `H · CX · H` block done **twice** in a row (an identity hiding in plain sight).

None of it changes the computation — it just bleeds fidelity. You're going to strip it out
**twice**: first by hand, to prove you understand *why* it's removable, then with a
transpiler that does the same reasoning automatically.

<b>How big is the waste?</b> Print the raw circuit's depth and CNOT count so you have a
baseline to beat.

In [ ]:
# your code here
???

<b>Simplify it yourself first, before reaching for any tool.</b> Work the identities above
and figure out what the circuit *actually* computes. Two observations crack it:

* The first block is a CNOT ladder with `H·H` and `X·X` pairs wrapped around it — the pairs
  vanish, leaving a plain ladder of CNOTs running outward from qubit 0.
* Every layer of the second block is secretly the identity: `CX·CX` cancels,
  `Rz(0.37)·Rz(-0.37)` cancels, and `H·CX·H·H·CX·H` collapses (the inner `H·H` goes, then
  the bare `CX·CX` goes).

So underneath all 49 CNOTs, the patient is just a **GHZ-state preparation**.

<b>Build that minimal circuit by hand and prove it's the same.</b> Make `by_hand` — an `H`
on qubit 0, then a CNOT ladder `0→1→2→…→7` — confirm `Operator(by_hand).equiv(Operator(RAW))`,
and print its CNOT count.

In [ ]:
by_hand = QuantumCircuit(N)
# your code here
???

Nicely done — the 49-CNOT monster was a GHZ preparation all along. But you don't want to
hand-analyze every circuit you'll ever run, and a transpiler does this reasoning for you
(and catches identities you might miss).

<b>Let the transpiler optimize, and confirm it lands where you did.</b> Run
`transpile(RAW, optimization_level=3, basis_gates=["u", "cx"])` to get `OPT`, check
`Operator(OPT).equiv(Operator(RAW))` is `True`, and print its depth and CNOT count — it
should match your by-hand result.

In [ ]:
# your code here
???

That should be a brutal reduction (≈49 CNOTs → ≈7). Now watch what those deleted gates
were costing you. We'll score by **state fidelity** to the ideal GHZ — the overlap of the
noisy output with the state we wanted.

<b>Compare raw vs. optimized at today's noise.</b> For each circuit, get its noisy density
matrix with the provided `noisy_state`, then `state_fidelity(rho, GHZ)`. Print both and the
ratio.

In [ ]:
GHZ = Statevector(OPT)        # the ideal target state

def noisy_state(circuit, s):  # provided
    qc = circuit.copy(); qc.save_density_matrix()
    t = transpile(qc, _dm, optimization_level=0,
                  basis_gates=["u", "cx", "save_density_matrix"])
    return DensityMatrix(_dm.run(t, noise_model=make_noise_model(s)).result().data()["density_matrix"])

# your code here
???

On today's 1%-per-CNOT hardware that's roughly a **2× jump** — from a coin-flip (≈0.53) to
usable (≈0.92) — for free, just by deleting redundant gates. And the win *grows* with the
noise: fidelity falls off like $(1-p)^{\text{gates}}$, so going $49 \to 7$ CNOTs matters far
more on a worse machine (at a 10%-error device it's $0.9^{49}\!\approx\!0.006$ vs
$0.9^{7}\!\approx\!0.48$, an ~80× gap). That's the Lab 9 lesson in one move: **the cheapest
mitigation is the gate you never run.** From here on we always work with `OPT`.

## 3. Pauli twirling (a callback to Lab 18)

Optimization removed *wasted* gates, but the 7 CNOTs that remain are still noisy. The
mitigations that follow (especially ZNE in §5) assume the leftover noise is **stochastic**
— random Pauli flips — not **coherent** (a systematic over-rotation that the same way every
time). Real hardware has both, and coherent error is the kind that quietly biases your
answer.

**Pauli twirling**, from Lab 18, fixes this: wrap each CNOT in random Paulis (chosen so the
*logical* circuit is unchanged) and average over many random versions. Coherent errors get
scrambled into stochastic ones; the average is unbiased. The price is that you now run an
**ensemble** of circuits instead of one.

<b>Build a twirled ensemble and prove it's still the same circuit.</b> Use the provided
`pauli_twirl(circuit, seed)` to make `n = 12` twirled copies of `OPT`, and check each one is
equivalent to `OPT` with `Operator(...).equiv(...)`. (They run *differently* on noisy
hardware but compute the same thing.)

In [ ]:
def pauli_twirl(circuit, seed):       # provided
    rng = np.random.default_rng(seed)
    cxc = Clifford(CXGate())                              # CX: qubit0=control, qubit1=target
    out = QuantumCircuit(circuit.num_qubits)
    for instr in circuit.data:
        op = instr.operation
        if op.name == "cx":
            c, t = [circuit.find_bit(q).index for q in instr.qubits]
            pc, pt = "IXYZ"[rng.integers(4)], "IXYZ"[rng.integers(4)]   # random before-Paulis
            after = Pauli(pt + pc).evolve(cxc, frame="s")               # = CX (pt⊗pc) CX
            lbl = after.to_label().replace("-", "").replace("i", "")
            a_t, a_c = lbl[0], lbl[1]                                   # label[0]=target side
            if pc != "I": getattr(out, pc.lower())(c)                   # before
            if pt != "I": getattr(out, pt.lower())(t)
            out.cx(c, t)
            if a_c != "I": getattr(out, a_c.lower())(c)                 # after
            if a_t != "I": getattr(out, a_t.lower())(t)
        else:
            out.append(op, instr.qubits)
    return out

# your code here
???

## 4. Readout-error mitigation

Even a perfectly prepared state gets corrupted *at the very end*, when each qubit is
measured: with probability `readout_rate(s)` a `0` is reported as a `1` or vice versa. For
our parity observable this is easy to model and to undo. Independent bit-flips with rate
$p$ shrink the parity expectation by a factor $(1-2p)$ **per qubit**:

$$\langle O\rangle_{\text{measured}} = (1-2p)^{N}\,\langle O\rangle_{\text{true}}.$$

So if you measure the flip rate $p$ (a quick calibration: prepare a known bit, see how
often it comes back wrong), you can divide it back out.

<b>Write the readout-mitigation factor.</b> Implement `readout_factor(s)` returning
$(1-2p)^N$ with $p =$ `readout_rate(s)`, and confirm it shrinks toward 0 as the hardware
gets noisier. Dividing a measured value by this factor removes the readout bias.

In [ ]:
def readout_factor(s):
    # your code here
    ???

for s in (1.0, 0.1, 0.01):
    print(f"s={s:<5} readout damps <O> by x{readout_factor(s):.3f}")

## 5. Zero-noise extrapolation

Here is the cleverest trick. You can't make the hardware *less* noisy on demand — but you
can make it *more* noisy on purpose, in a controlled way, and then extrapolate back to the
noise-free point you actually want.

The control knob is **gate folding**: replacing a circuit $U$ by $U U^\dagger U$ runs the
same computation but with **3×** the gates, hence ~3× the noise. Do it again for 5×, and so
on. Measure $\langle O\rangle$ at noise scales $1, 3, 5$, fit a line, and read off its value
at scale **0** — the zero-noise estimate.

<b>Implement folding.</b> Write `fold(prep, k)` that returns `prep` followed by `k`
repetitions of (`prep.inverse()` then `prep`). `k=0` is the bare circuit (scale 1), `k=1` is
scale 3, `k=2` is scale 5.

In [ ]:
def fold(prep, k):
    # your code here
    ???

print("scale 1 CNOTs:", fold(OPT, 0).count_ops().get("cx", 0))
print("scale 3 CNOTs:", fold(OPT, 1).count_ops().get("cx", 0))
print("scale 5 CNOTs:", fold(OPT, 2).count_ops().get("cx", 0))

<b>Extrapolate to zero noise.</b> Write `zne(prep, s)` that measures `expval(fold(prep, k), s)`
at `k = 0, 1, 2` (noise scales `[1, 3, 5]`), fits a straight line with `np.polyfit(scales,
values, 1)`, and returns its value at scale `0`. Compare the bare value (scale 1) to the
ZNE estimate at a cleaner `s = 0.05`.

In [ ]:
def zne(prep, s):
    # your code here
    ???

s = 0.05
print(f"bare <O> (scale 1): {expval(OPT, s):.4f}")
print(f"ZNE estimate      : {zne(OPT, s):.4f}")
print(f"ideal             : {IDEAL:.4f}")

## 6. Circuit knitting (and why it isn't free)

One more idea, because two-qubit gates are the expensive part: what if you could **cut** an
entangling gate and run the circuit as two smaller, less-entangled pieces? That's **circuit
knitting** (gate cutting). The catch is the bill: each cut gate is replaced by a sum of
local operations you must sample, and the number of samples needed grows by a factor
$\gamma^2$ **per cut**. For a single CNOT, $\gamma = 3$, so one cut costs **9×** the
sampling — and cuts *multiply*, so two cuts cost 81×. It's a real tool, but the cost is
exponential in the number of cuts.

The cell below cuts one CNOT of a small circuit and reports the overhead. <b>Compute the
total sampling overhead $\gamma^2$.</b>

In [ ]:
from qiskit.quantum_info import PauliList
from qiskit_addon_cutting import partition_problem, generate_cutting_experiments

demo = QuantumCircuit(4); demo.h(0)
for i in range(3): demo.cx(i, i + 1)
parts = partition_problem(circuit=demo, partition_labels="AABB",
                          observables=PauliList(["ZZZZ"]))
gamma = float(np.prod([b.overhead for b in parts.bases]))   # provided: per-cut gammas
subexp, _ = generate_cutting_experiments(circuits=parts.subcircuits,
                                         observables=parts.subobservables, num_samples=np.inf)

# your code here
???
print("subcircuits:", list(parts.subcircuits.keys()),
      "| total subexperiments:", sum(len(v) for v in subexp.values()))

## 7. The resource model: what does an answer cost?

You now have a toolbox. The honest question on real hardware is never "can I be perfect?"
— it's "what does it **cost** to be good enough?" We'll fix "good enough" as

$$|\langle O\rangle_{\text{mitigated}} - \langle O\rangle_{\text{ideal}}| \le 0.02.$$

and charge you on three axes that come from the hardware level `s` and the techniques you
turn on:

* **dollars** — a steep build cost for cleaner hardware (`∝ s^{-0.7}`), **plus** the cost of
  the runtime you spend on mitigation.
* **runtime** — more circuits (ZNE folds, twirl copies) *and* more shots take longer, and
  noisier hardware needs **more shots** to pull $\langle O\rangle$ out of the statistics.
* **years until it exists** — cleaner hardware is further in the future.

The two functions below wire all of that up. `run_experiment` applies the techniques you
choose and returns the mitigated $\langle O\rangle$; `estimate_resources` returns the bill.

In [ ]:
# Provided engine — you do not need to change this.
def run_experiment(s, optimize=True, readout=False, zne_on=False):
    prep = OPT if optimize else RAW
    val = zne(prep, s) if zne_on else expval(prep, s)
    if not readout:
        val = val * readout_factor(s)        # uncorrected readout bias
    return val

def estimate_resources(s, optimize=True, readout=False, zne_on=False, twirl=False):
    build = 3e6 * (1 / s) ** 0.7             # cleaner hardware costs more to build...
    years = 10.0 * (-np.log10(s))            # ...and is further in the future
    overhead = 1.0                           # one circuit
    if readout: overhead += 0.5              # + calibration circuits
    if zne_on:  overhead *= 3                # 3 noise scales to fold + measure
    if twirl:   overhead *= 12               # 12 twirled copies to average
    shots = (s / 0.005) ** 0.5               # noisier hardware needs MORE shots to resolve <O>
    runtime = overhead * shots
    dollars = build + runtime * 1e6          # the runtime is billed too
    return {"dollars": dollars, "runtime_s": runtime, "years": years}

def report(s, **strat):
    val = run_experiment(s, optimize=strat.get("optimize", True),
                         readout=strat.get("readout", False),
                         zne_on=strat.get("zne_on", False))
    err = abs(IDEAL - val)
    res = estimate_resources(s, **strat)
    ok = "WIN " if err <= 0.02 else "miss"
    print(f"[{ok}] s={s:<7g} <O>={val:.3f} err={err:.3f} | "
          f"${res['dollars']/1e6:5.1f}M  {res['runtime_s']:4.1f}s  {res['years']:4.1f}yr  {strat}")
    return err <= 0.02, res

<b>Get a feel for the trade-offs.</b> Call `report(...)` on a few combinations and watch
the WIN/miss flag and the bill move. Try at least: today with everything
(`report(1.0, readout=True, zne_on=True, twirl=True)`), a mid machine with light mitigation
(`report(0.05, readout=True)`), and a clean machine with nothing extra
(`report(0.01)`).

In [ ]:
# your code here
???

## 8. Find a deal you can live with

Today's hardware (`s = 1`) can't hit the target no matter how much you mitigate — you have
to buy *some* improvement. But how much, and paired with which techniques, depends on what
you're optimizing for. There is no single best answer: the **cheapest**, the **fastest**,
and the **soonest** winning configurations are three different machines.

<b>Hunt for winners.</b> Scan a few hardware levels `s` and technique combinations, keep the
ones that hit the 0.02 target, and pick the one **you** are happiest with — minimizing
dollars, runtime, years, or your own blend. Use `report(...)` (it returns
`(is_win, resources)`), collect the winners, and print your favorite with a sentence saying
why.

In [ ]:
# your code here
???

The point isn't a single number — it's the **shape** of the choice. Cleaner hardware is
expensive and far off but needs little mitigation; noisier hardware is here sooner and
cheaper but you pay in runtime to mitigate it back into spec; and skipping the optimization
in §2 makes *everything* harder. That is real quantum resource estimation: not "is it
perfect," but "what's the cheapest road to good enough."

## Wrap-up

You implemented a full mitigation stack by hand — **optimize, twirl, correct readout,
extrapolate** — and saw circuit knitting's exponential price. The throughline from Lab 9 is
now explicit: **accuracy is bought with a resource**, and the engineer's job is to spend the
*right* one. Lab 18 made the noise well-behaved; this lab made it *affordable*; next, **Lab
20** asks whether you can pay once, up front, to remove the error structurally — quantum
error *correction*.